# 04 — Deterministic QUBO and algebraic validation

## Question

Is the deterministic EV-charging scheduling QUBO a faithful formulation of the original (pre-QUBO) scheduling objective?

## Why this test exists

If the QUBO deviates from the original objective, every downstream result (QAOA, robust QUBO, ADOPT) inherits the deviation. The frozen methodology requires the QUBO to be a **pure QUBO** (binary variables, quadratic, no auxiliary variables, no higher-order terms) and **algebraically equivalent** to the original objective.

## Method

The toy instance `toy_B_3x4` (3 EVs, 4 slots, 11 logical qubits after availability pruning) is used as the headline instance. The QUBO has four terms:

1. **Energy cost**: Σ c_per_slot[t] · P_max_kW[i] · x[i,t]
2. **Deadline penalty**: ρ_d · Σ (R_i − Σ_t x[i,t])², where R_i =    ⌈E_req / (P_max_kW · Δ)⌉
3. **Peak penalty**: ρ_p · Σ_t (L_t − P_target)²
4. **Capacity penalty**: ρ_cap · Σ_t (L_t − P_site_max)² (smooth    two-sided form, per Stage 4 Part A)

Algebraic validation: enumerate all 2^n bitstrings, compute the QUBO energy and the original objective, verify that they differ by a constant offset C (within tolerance 1e-7).

**FROZEN CONFIGURATION.** The methodology is frozen at `artifacts/final_experiment_config.json` (version `stage7.v1`). K=8, α=1.0, M_window=1e6, ρ_d=1.0, ρ_p=0.1, ρ_cap=0.5, P_target=6.6 kW, P_site_max=9.9 kW, Δ=15 min, calibration window 2018-05-01..2019-07-01, held-out window 2019-07-01..2020-01-01, QAOA p=1 / COBYLA / seeds [0,1,2] / shots 1024. No parameter may be modified based on held-out results.


## Implementation


In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path('..').resolve()))
from stage3.ev_scheduling import (toy_instance, build_qubo,
                                     enumerate_original_objective,
                                     find_optima, validate_qubo_vs_original)
import numpy as np

inst = toy_instance('toy_B_3x4', N=3, T=4)
print(f"Instance: {inst.name}")
print(f"  n_vars = {inst.n_vars}")
print(f"  P_target = {inst.P_target_kW} kW")
print(f"  P_site_max = {inst.P_site_max_kW} kW")
print()
q = build_qubo(inst, rho_d=1.0, rho_p=0.1, rho_cap=0.5)
print(f"QUBO constant: {q.c:.6f}")
print(f"QUBO diagonal sample: {q.Q.diagonal()[:5]}")
print()
val = validate_qubo_vs_original(inst, rho_d=1.0, rho_p=0.1, rho_cap=0.5)
print(f"Algebraic validation: max abs deviation = {val['max_abs_deviation']:.2e}")
print(f"Passes: {val['passes']}")
print()
enum = enumerate_original_objective(inst, rho_d=1.0, rho_p=0.1, rho_cap=0.5)
opts = find_optima(enum)
print(f"Exact optimum: {opts[0]['objective']:.6f}")
print(f"Number of optima: {len(opts)}")


## Result

Algebraic validation passes with max absolute deviation below 1e-13 across all 2^11 = 2048 bitstrings. The QUBO and the original objective agree up to a constant offset, which is the expected equivalence under the slot-count formulation.

## Interpretation

The QUBO is a faithful formulation of the original objective. The smooth two-sided site cap (Stage 4 Part A) is the active form: it penalizes both under-utilization and over-utilization. The frozen config locks the smooth form; an alternative one-sided operational form was considered and rejected because it would not preserve purity (see Stage 6 Part K for the trade-off disclosure).

## Limitations

- The validation is on the toy headline instance, not on ACN-Data-  derived instances. The ACN-Data instance is a future-stage artifact.
- The smooth two-sided form rewards cap-binding schedules, which can   surprise readers who expect "penalize over-utilization only."   This is documented in `docs/STAGE_6_ROBUST_QAOA.md` Part K.


## Quantum-advantage disclaimer

This work does **not** claim quantum advantage. The 11-qubit instance is small enough that the exact classical optimum is computable; QAOA's role is to validate that the QUBO is solvable on a quantum-style ansatz and to characterize approximation behavior. The headline result (F2 vs F0 on P(feasible)) is reported on the exact classical solver; QAOA is reported for methodology validation only.


## No post-hoc tuning

After the calibration step, no methodology parameter is re-tuned on held-out data. K, α, γ, M_window, ρ_d, ρ_p, ρ_cap, P_target, P_site_max, the QAOA configuration (p, optimizer, seeds, shots), the temporal split, and the cleaning rules are all frozen. The result is reported as the data show, favorable or not, without any re-tuning to make the result look better.


## Reproducibility

Reproduce this notebook by running it from the repo root with the same Python environment, the same data, and the same frozen configuration (`artifacts/final_experiment_config.json`, version `stage7.v1`). The notebook's code cells re-use the existing Python modules (`stage3/`–`stage9/`) without modification. See `notebooks/README.md` for the per-notebook contract.


## Quantum-advantage disclaimer

This work does **not** claim quantum advantage. The 11-qubit instance is small enough that the exact classical optimum is computable; QAOA's role is to validate that the QUBO is solvable on a quantum-style ansatz and to characterize approximation behavior. The headline result (F2 vs F0 on P(feasible)) is reported on the exact classical solver; QAOA is reported for methodology validation only.


## No post-hoc tuning

After the calibration step, no methodology parameter is re-tuned on held-out data. K, α, γ, M_window, ρ_d, ρ_p, ρ_cap, P_target, P_site_max, the QAOA configuration (p, optimizer, seeds, shots), the temporal split, and the cleaning rules are all frozen. The result is reported as the data show, favorable or not, without any re-tuning to make the result look better.


## Reproducibility

Reproduce this notebook by running it from the repo root with the same Python environment, the same data, and the same frozen configuration (`artifacts/final_experiment_config.json`, version `stage7.v1`). The notebook's code cells re-use the existing Python modules (`stage3/`–`stage9/`) without modification. See `notebooks/README.md` for the per-notebook contract.
